<a href="https://colab.research.google.com/github/luke-wellerman/luke-wellerman-ds2002-fa26/blob/main/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:

df['revenue'] = df['price']*df['qty']
print("The sum of revenue (price * quantity) for all vendors combinded: ", sum(df['revenue']))
print("Total units sold:", df['qty'].sum())


The sum of revenue (price * quantity) for all vendors combinded:  8520.0
Total units sold: 783


Calcualted the total revenue and total units sold

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:

by_category = (df.groupby('category', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False))
by_category['share'] = (by_category['revenue'] / by_category['revenue'].sum() * 100)
by_category

,category,revenue,share
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


groups by category and finds the total revenue for that category, allowing a reference to find share percentage

Took each revenue, divided it by the total revenue to get fraction of share, multiplied by 100 to get percentages.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:

print('V-01: ', 'Orders: ', len(df[df['vendor_id'] == 'V-01']), ' | Average Revenue: ', df.loc[df['vendor_id'] == 'V-01', 'revenue'].sum()/len(df[df['vendor_id'] == 'V-01']))
print('V-05: ', 'Orders: ', len(df[df['vendor_id'] == 'V-05']), ' | Average Revenue: ', df.loc[df['vendor_id'] == 'V-05', 'revenue'].sum()/len(df[df['vendor_id'] == 'V-05']))
print('V-10: ', 'Orders: ', len(df[df['vendor_id'] == 'V-10']), ' | Average Revenue: ', df.loc[df['vendor_id'] == 'V-10', 'revenue'].sum()/len(df[df['vendor_id'] == 'V-10']))
print('V-18: ', 'Orders: ', len(df[df['vendor_id'] == 'V-18']), ' | Average Revenue: ', df.loc[df['vendor_id'] == 'V-18', 'revenue'].sum()/len(df[df['vendor_id'] == 'V-18']))

V-01:  Orders:  94  | Average Revenue:  22.595744680851062
V-05:  Orders:  93  | Average Revenue:  20.580645161290324
V-10:  Orders:  105  | Average Revenue:  20.314285714285713
V-18:  Orders:  108  | Average Revenue:  21.75


Finds the number of orders when filtering by each vendor, and divides the total revenue by amount of orders to find average revenue per order.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
print(round(df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100,1), '%')

20.8 %


Finds where category = merch, sums the revenue and compares it to total revenue of the df as a percentage

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
unmatched = joined[joined['vendor_name'].isna()]
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown')

print("Rows:", len(df), ' | ', len(joined))
print("Revenue:", df['revenue'].sum(), ' | ', joined['revenue'].sum())


Rows: 400  |  400
Revenue: 8520.0  |  8520.0


**The unmatched vendor, and what I did about it:** The unmatched vendor was V-18 and I kept the data while replacing the null value with a lable of unknown to indicate limitation in the data while ensuring all data remains.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = joined.pivot_table(index='vendor_name', columns='category',
                           values='revenue', aggfunc='sum',
                           margins=True, margins_name='Total', fill_value=0)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


Learned how to creat pivot table inluding the name and categorys while making the values displayed the revenue and adding on the final row/col a total.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A)I would reccomend vendors to enhance their supply of food in place of rain gear since the revenue from rain gear(901.5) is much less than the revenue from food(4293.0). With this decrease in rain gear availible, it would also be responisible to increase the price of rain gear in an attempt to increase rain gear revenue in times when rain is present. Additionaly, since merchandise accounts for 20% of revenue, I would reccomend analyzing the prices of items in that category and determine if a change in price could lead to more revenue. Increasing the price may lead to the same quantity purchased with an increase in revenue. Decreasing the price may cause an increase in quantity sold, leading to the possibility of additional revenue.


B) One large issue with the data is Q6. The data with vendors unknown accounts for the largest portion of revenue when compared to other vendors. If the data is unrealiable or false since the vendor is not listed, it should not be included in making conclusions based on the data.